In [ ]:
import os
import re
from pathlib import Path
from textwrap import dedent
from typing import Dict, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import spearmanr

from utils import kernel_mean_quantile_band_rankx, load_paths

# Machine-specific data pointers live in paths.yaml, which is gitignored.
# Keep notebooks portable by reading local dataset locations from that file.
PATHS = load_paths()

hf_path = PATHS["hf_predictions"]
df = pd.read_csv(hf_path)
echo_path = PATHS["ahus_echo"]
echo = pd.read_csv(echo_path)
fnr_map_path = PATHS["ahus_echo_fnr_map"]
fnr_map = pd.read_csv(fnr_map_path)

echo = echo.merge(
    fnr_map[["FOEDSELSNR_SINGLE", "FOEDSELSNR_DOUBLE"]],
    left_on="ID",
    right_on="FOEDSELSNR_SINGLE",
    how="left",
)
echo = echo[~echo["FOEDSELSNR_DOUBLE"].isna()]
echo = echo.rename(columns={"FOEDSELSNR_DOUBLE": "FOEDSELSNR"})
echo = echo.drop(columns=["FOEDSELSNR_SINGLE"])

MAX_DAYS = 3
N_GRID = 400
QS = (0.25, 0.75)

CACHE_DIR = "./cache/"
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

# Table outputs are written repeatedly by later analysis cells.
# Create both the project table directory and article table mirror up front.
TABLES_DIR = Path("tables")
ARTICLE_TABLES_DIR = Path("review/article/tables")
for _table_output_dir in (TABLES_DIR, ARTICLE_TABLES_DIR):
    _table_output_dir.mkdir(parents=True, exist_ok=True)

FIGURE_OUTPUT_DIRS = {fmt: Path("figures") / fmt for fmt in ("png", "pdf")}
for _figure_output_dir in FIGURE_OUTPUT_DIRS.values():
    _figure_output_dir.mkdir(parents=True, exist_ok=True)


def figure_output_path(filename: str) -> Path:
    return FIGURE_OUTPUT_DIRS[Path(filename).suffix.lstrip(".")] / filename

# set matplotlib C0 to Maroon
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["steelblue", "#ad3339"] + plt.rcParams["axes.prop_cycle"].by_key()["color"][1:]
)

# reproducibility
np.random.seed(42)
rng = np.random.default_rng(24)  # for reproducibility

# Define mapping from parameters to field names from the echo database

In [ ]:
CONFIG_PATH = Path("echo_settings.yaml")

with CONFIG_PATH.open("r") as f:
    cfg = yaml.safe_load(f)

COMPACS_MAPPING = cfg["COMPACS_MAPPING"]
UNITS_MAPPING = cfg["UNITS_MAPPING"]
COMORBIDITY_MAP = cfg["COMORBIDITY_MAP"]
PARAMETERS = cfg["PARAMETERS"]

echo_min_max_cutoff_mapping = {
    k: tuple(v) for k, v in cfg["MIN_MAX_CUTOFF_MAPPING"].items()
}

COMORBIDITY_REGEX = cfg["COMORBIDITY_REGEX"]
PLOT_LIMS = cfg["PLOT_LIMS"]

N_BOOTSTRAP = cfg["N_BOOTSTRAP"]

# Keep patients with both echo and ECG

In [ ]:
df_pids = df["FOEDSELSNR"]
echo_pids = echo["FOEDSELSNR"]
overlap = np.intersect1d(df_pids, echo_pids)
df = df[df["FOEDSELSNR"].isin(overlap)]
echo = echo[echo["FOEDSELSNR"].isin(overlap)]

# Backwards fill dataframe; the raw version contains one row per edit

In [ ]:
def first_available_column(df: pd.DataFrame, candidates):
    """Return the first column from candidates that exists in df, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def coalesce_series(df: pd.DataFrame, candidates):
    """
    Return a single Series coalescing across candidates (first non-null per row),
    using only columns present in df. Returns (series, cols_used).
    """
    avail = [c for c in candidates if c in df.columns]
    if not avail:
        return pd.Series(index=df.index, dtype=float), []
    s = df[avail].bfill(axis=1).iloc[:, 0]
    return s, avail


group_series = {}
chosen_columns = {}
for grp, candidates in COMPACS_MAPPING.items():
    s, used = coalesce_series(echo, candidates)
    group_series[grp] = s
    chosen_columns[grp] = used

meta_cols = ["SeriesDate", "FOEDSELSNR"]
for col in meta_cols:
    if col in echo.columns:
        group_series[col] = echo[col]

echo_hf_grouped = pd.DataFrame(group_series, index=echo.index)

In [ ]:
# Drop outliers in E and A, then compute E/A
E_lower, E_upper = echo_min_max_cutoff_mapping["E"]
A_lower, A_upper = echo_min_max_cutoff_mapping["A"]

E = echo_hf_grouped["E"].copy()
A = echo_hf_grouped["A"].copy()

E[(E < E_lower) | (E > E_upper)] = np.nan
A[(A < A_lower) | (A > A_upper)] = np.nan

echo_hf_grouped["E"] = E
echo_hf_grouped["A"] = A
echo_hf_grouped["E/A"] = E / A

echo_hf_grouped["BSA"] = echo_hf_grouped["CO"] / echo_hf_grouped["CI"]
echo_hf_grouped["LVEDVi"] = echo_hf_grouped["LVEDVOL"] / echo_hf_grouped["BSA"]

LAv_lower, LAv_upper = echo_min_max_cutoff_mapping["LAv"]
echo_hf_grouped["LAv"] = echo_hf_grouped["LAv"].where(
    (echo_hf_grouped["LAv"] >= LAv_lower) & (echo_hf_grouped["LAv"] <= LAv_upper)
)
echo_hf_grouped["LAVi"] = echo_hf_grouped["LAv"] / echo_hf_grouped["BSA"]
echo_hf_grouped["LVMi"] = echo_hf_grouped["LVM"] / echo_hf_grouped["BSA"]

echo_hf_grouped["e'"] = echo_hf_grouped["e'"] * 100  # m/s to cm/s

# Drop redundant/non-indexed volumes
echo_hf_grouped = echo_hf_grouped.drop(columns=["LAv", "LVEDVOL", "CO"])

# Filter outliers in echo dataframe

In [ ]:
echo_hf_grouped = echo_hf_grouped.copy()

for name in echo_min_max_cutoff_mapping.keys():
    if name not in echo_hf_grouped.columns:
        continue

    count_before = echo_hf_grouped[name].notna().sum()

    lower, upper = echo_min_max_cutoff_mapping.get(name, (None, None))
    echo_hf_grouped.loc[echo_hf_grouped[name] < lower, name] = np.nan
    echo_hf_grouped.loc[echo_hf_grouped[name] > upper, name] = np.nan

    count_after = echo_hf_grouped[name].notna().sum()

    print(f"{name:<12}: {count_before:,} -> {count_after:,}")

In [ ]:
echo_hf_grouped["SeriesDate"] = pd.to_datetime(
    echo_hf_grouped["SeriesDate"], errors="coerce"
)
df["time"] = pd.to_datetime(df["time"], errors="coerce")

echo_hf_grouped = echo_hf_grouped.sort_values(by=["FOEDSELSNR", "SeriesDate"])
echo_hf_grouped.update(echo_hf_grouped.groupby("FOEDSELSNR").bfill())


print(len(echo_hf_grouped))
echo_first = (
    echo_hf_grouped.dropna(subset=["FOEDSELSNR", "SeriesDate"])
    .sort_values(["FOEDSELSNR", "SeriesDate"])
    .drop_duplicates(subset=["FOEDSELSNR"], keep="first")
    .reset_index(drop=True)
)
print(len(echo_first))


df_times = df.dropna(subset=["FOEDSELSNR", "time"])[["FOEDSELSNR", "time"]].copy()

cand = echo_first[["FOEDSELSNR", "SeriesDate"]].merge(
    df_times, on="FOEDSELSNR", how="left"
)

cand["abs_diff"] = (cand["time"] - cand["SeriesDate"]).abs()

idx = (
    cand.dropna(subset=["time"])  # only patients with at least one ECG
    .groupby("FOEDSELSNR")["abs_diff"]
    .idxmin()
)

nearest = cand.loc[idx].copy()

no_ecg_patients = set(echo_first["FOEDSELSNR"]) - set(nearest["FOEDSELSNR"])
if no_ecg_patients:
    nearest = pd.concat(
        [
            nearest,
            echo_first.loc[
                echo_first["FOEDSELSNR"].isin(no_ecg_patients),
                ["FOEDSELSNR", "SeriesDate"],
            ].assign(time=pd.NaT, abs_diff=pd.NaT),
        ],
        ignore_index=True,
    )

matched = nearest.merge(
    echo_first, on=["FOEDSELSNR", "SeriesDate"], how="left", suffixes=("", "_echo")
)

matched["delta"] = matched["time"] - matched["SeriesDate"]
matched["delta_days"] = matched["delta"].dt.total_seconds() / (24 * 3600)
matched["abs_delta_days"] = matched["delta_days"].abs()

print(f"First-echo patients: {len(echo_first)}")
print(f"Matched with at least one ECG: {matched['time'].notna().sum()}")
print(f"No ECG found: {matched['time'].isna().sum()}")

In [ ]:
td_limit = pd.Timedelta(days=MAX_DAYS)

df_times = df.dropna(subset=["FOEDSELSNR", "time"])[
    ["FOEDSELSNR", "time", "ensemble", "age", "sex", "hf_icd_flag_ever"]
].copy()

cand = echo_first[["FOEDSELSNR", "SeriesDate"]].merge(
    df_times, on="FOEDSELSNR", how="left"
)
cand["abs_diff"] = (cand["time"] - cand["SeriesDate"]).abs()

idx = cand.dropna(subset=["time"]).groupby("FOEDSELSNR")["abs_diff"].idxmin()

nearest = cand.loc[idx].copy()

no_ecg_patients = set(echo_first["FOEDSELSNR"]) - set(nearest["FOEDSELSNR"])
if no_ecg_patients:
    nearest = pd.concat(
        [
            nearest,
            echo_first.loc[
                echo_first["FOEDSELSNR"].isin(no_ecg_patients),
                ["FOEDSELSNR", "SeriesDate"],
            ].assign(
                time=pd.NaT, abs_diff=pd.NaT, ensemble=np.nan, age=np.nan, sex=np.nan
            ),
        ],
        ignore_index=True,
    )

matched = nearest.merge(
    echo_first, on=["FOEDSELSNR", "SeriesDate"], how="left", suffixes=("", "_echo")
)

matched["delta"] = matched["time"] - matched["SeriesDate"]
matched["delta_days"] = matched["delta"].dt.total_seconds() / (24 * 3600)
matched["abs_delta_days"] = matched["delta_days"].abs()

keep_mask = matched["delta"].notna() & (matched["delta"].abs() <= td_limit)
kept = matched.loc[keep_mask].copy()
kept["age"] = kept["age"].str.extract(r"(\d+)").astype(int)

# drop age < 18 and age > 100 and print then umber of dropped patients
age_mask = (kept["age"] >= 18) & (kept["age"] <= 100)
dropped_age = len(kept) - age_mask.sum()
if dropped_age > 0:
    print(f"Dropping {dropped_age} patients due to age < 18 or > 100")
kept = kept.loc[age_mask]
sex_map = {"M": 1.0, "F": 0.0, "O": 1.0}
kept["sex"] = kept["sex"].map(sex_map).fillna(1.0).astype(float)

In [ ]:
print(f"Total first-echo patients: {len(matched)}")
print(f"Kept within ±{MAX_DAYS} days: {len(kept)}")
print(f"Dropped: {len(matched) - len(kept)}")

s = kept["delta_days"]
if not s.empty:
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.hist(-s, bins=int(MAX_DAYS * 24 + 1), edgecolor="C0")
    ax.set_xlabel(f"Time from ECG to echocardiography (days)")
    ax.set_ylabel("Patients")
    fig.tight_layout()
    ax.set_xticks(np.arange(-int(MAX_DAYS), int(MAX_DAYS) + 1, 1))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_yticks([1000, 2000, 3000], labels=["1k", "2k", "3k"])
    plt.savefig(figure_output_path("ecg_echo_time_difference_histogram.png"), dpi=300)
    plt.savefig(figure_output_path("ecg_echo_time_difference_histogram.pdf"), bbox_inches="tight")
    plt.show()

    mean_abs_diff = s.abs().mean()
    print(
        f"Mean absolute time difference between ECG and Echo: {mean_abs_diff*24:.2f} hours"
    )
    # count how often before vs affter
    before_count = (s < 0).sum()
    after_count = (s > 0).sum()
    print(f"ECG before Echo: {before_count} times")
    print(f"ECG after Echo: {after_count} times")

else:
    print("No pairs within the specified time window.")

In [ ]:
cache_file_name = "diagnoses.csv"
if not os.path.exists(os.path.join(CACHE_DIR, cache_file_name)):
    diag = pd.read_csv(PATHS["ahus_diagnoses"], dtype="str")
    demo = pd.read_csv(PATHS["ahus_demography"], dtype="str")
    demo_pids = demo["PERSONID"].unique().tolist()
    diag_pids = diag["PERSONID"].unique().tolist()
    common_pids = set(demo_pids).intersection(set(diag_pids))
    demo_foedselsnrs = demo["FOEDSELSNR"].unique().tolist()
    echo_foedselsnrs = kept["FOEDSELSNR"].unique().tolist()
    common_foedselsnrs = set(demo_foedselsnrs).intersection(set(echo_foedselsnrs))
    subdemo = demo[demo["FOEDSELSNR"].isin(common_foedselsnrs)].copy()
    subdiag = diag[diag["PERSONID"].isin(subdemo["PERSONID"].unique())].copy()
    # add the FOEDSELSNR to subdiag by merging on PERSONID
    subdiag = subdiag.merge(
        subdemo[["PERSONID", "FOEDSELSNR"]], on="PERSONID", how="left"
    )

    # endow subdiag with ecg_date, taken from the "kept" dataframe
    subdiag = subdiag.merge(
        kept[["FOEDSELSNR", "time"]].drop_duplicates(), on="FOEDSELSNR", how="left"
    )
    subdiag.to_csv(os.path.join(CACHE_DIR, cache_file_name), index=False)
else:
    subdiag = pd.read_csv(os.path.join(CACHE_DIR, cache_file_name), dtype="str")
subdiag["DIAGNOSETID"] = pd.to_datetime(subdiag["DIAGNOSETID"])
subdiag["time"] = pd.to_datetime(subdiag["time"])

# from "df", add the age and sex columns to subdiag by merging on PERSONID
df["age"] = pd.to_numeric(df["age"].str.replace("Y", ""))
df["sex"] = df["sex"].str.lower().fillna("male")
subdiag = subdiag.merge(df[["PERSONID", "age", "sex"]], on="PERSONID", how="left")

subdiag["IS_BASELINE"] = (
    subdiag["DIAGNOSETID"] <= subdiag["time"] + pd.Timedelta(days=-1)
).astype(int)

In [ ]:
for comorbidity, patterns in COMORBIDITY_REGEX.items():
    kept[comorbidity] = 0
    mask = subdiag["KODE"].str.match("|".join(patterns))
    affected_foedselsnrs = subdiag.loc[mask, "FOEDSELSNR"].unique().tolist()
    kept.loc[kept["FOEDSELSNR"].isin(affected_foedselsnrs), comorbidity] = 1

    print(f"Comorbidity '{comorbidity}': {kept[comorbidity].sum()} patients")

In [ ]:
# Ensure table output directory exists if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

plot_cols = [c for c in PARAMETERS if c not in ("sex", "age")]

kept_filtered = kept.copy()


corr_results = []
for col in [c for c in plot_cols if c != "sex"]:
    if col == "FOEDSELSNR":
        continue

    x = kept_filtered[col].to_numpy()
    y = kept_filtered["ensemble"].to_numpy()

    # drop NaNs for the pair
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = x.size

    if n < 3:
        continue

    corr, _ = spearmanr(x, y)

    boot_corrs = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)
        bx = x[idx]
        by = y[idx]
        bcorr, _ = spearmanr(bx, by)
        if np.isfinite(bcorr):
            boot_corrs.append(bcorr)

    boot_corrs = np.asarray(boot_corrs)
    ci_low = np.percentile(boot_corrs, 2.5)
    ci_high = np.percentile(boot_corrs, 97.5)

    corr_results.append((col, n, corr, ci_low, ci_high))

corr_df = pd.DataFrame(
    corr_results,
    columns=["parameter", "n", "spearman_corr", "ci_low", "ci_high"],
)

plot_cols = sorted(
    plot_cols,
    key=lambda c: (
        abs(corr_df.loc[corr_df["parameter"] == c, "spearman_corr"].values[0])
        if (corr_df["parameter"] == c).any()
        else -np.inf
    ),
    reverse=True,
)

corr_df = (
    corr_df.assign(abs_corr=lambda d: d["spearman_corr"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

rows = []
for _, row in corr_df.iterrows():
    rows.append(
        "      "
        f"{row['parameter']} & "
        f"\\num{{{int(row['n'])}}} & "
        f"\\num{{{row['spearman_corr']:.2f}}} & "
        f"\\num{{{row['ci_low']:.2f}}} & "
        f"\\num{{{row['ci_high']:.2f}}} \\\\"
    )
body = "\n".join(rows) + "\n"

tabular = (
    r"   {\footnotesize"
    "\n   \\begin{tabular*}{\\linewidth}{@{\\extracolsep{\\fill}}lrrrr@{}}\n"
    "      \\toprule\n"
    "      Parameter & $n$ & $\\rho$ & "
    " \\multicolumn{2}{c}{95\\% CI} \\\\\n"
    "      & & & Low & High \\\\\n"
    "      \\midrule\n"
    f"{body}"
    "      \\bottomrule\n"
    "   \\end{tabular*}\n"
    "   }\n"
)

latex = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between AI–ECG HF score and echocardiographic parameters, calculated on the Ahus cohort.}\n"
    "   \\label{tab:spearman_corr}\n"
    f"{tabular.replace('_', ' ')}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item $n$ is the number of pairs used for each correlation. "
    "Higher absolute $\\rho$ indicates stronger monotonic association. "
    "Values are 95\\% bootstrap confidence intervals (\\num{10000} resamples).\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

with open(TABLES_DIR / "spearman_table.tex", "w") as f:
    f.write(latex)

In [ ]:
# Ensure table output directory exists if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

def compute_corr_for_df(df, param_cols, target_col="ensemble"):
    out = []
    for col in param_cols:
        if col in ("sex", "FOEDSELSNR"):
            continue
        corr, _ = spearmanr(df[col], df[target_col], nan_policy="omit")
        out.append((col, corr))
    return pd.DataFrame(out, columns=["parameter", "rho"])


overall_corr_df = compute_corr_for_df(kept_filtered, plot_cols)

# sort by |corr| overall
overall_corr_df = (
    overall_corr_df.assign(abs_corr=lambda d: d["rho"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

param_order = overall_corr_df["parameter"].tolist()
pretty_names = overall_corr_df["parameter"].tolist()

lvef_groups = {
    r"$\leq$ 40\%": kept_filtered["LVEF"] <= 40,
    r"41–49\%": (kept_filtered["LVEF"] > 40) & (kept_filtered["LVEF"] < 50),
    r"$\geq$ 50\%": kept_filtered["LVEF"] >= 50,
}

lvef_corrs = {}
for label, mask in lvef_groups.items():
    df_sub = kept_filtered.loc[mask]
    rhos = []
    for col in param_order:
        corr, _ = spearmanr(df_sub[col], df_sub["ensemble"], nan_policy="omit")
        rhos.append(corr)
    lvef_corrs[label] = rhos

sex_values = [v for v in kept_filtered["sex"].dropna().unique()]

sex_label_map = {val: f"sex = {val}" for val in sex_values}

sex_corrs = {}
for val in sex_values:
    label = sex_label_map[val]
    df_sub = kept_filtered.loc[kept_filtered["sex"] == val]
    rhos = []
    for col in param_order:
        corr, _ = spearmanr(df_sub[col], df_sub["ensemble"], nan_policy="omit")
        rhos.append(corr)
    sex_corrs[label] = rhos


def perm_test_diff_spearman_overall(df, param, target="ensemble", B=1000, seed=123):
    rng = np.random.default_rng(seed)

    sub = df[[param, target, "sex"]].dropna()

    m = sub[sub["sex"] == 1.0]
    f = sub[sub["sex"] == 0.0]

    if len(m) < 4 or len(f) < 4:
        return np.nan

    rho_m, _ = spearmanr(m[param], m[target])
    rho_f, _ = spearmanr(f[param], f[target])
    obs_diff = rho_m - rho_f

    diffs = np.empty(B)

    for b in range(B):
        permuted = sub.copy()
        permuted["sex"] = rng.permutation(permuted["sex"].values)

        m_p = permuted[permuted["sex"] == 1.0]
        f_p = permuted[permuted["sex"] == 0.0]

        if len(m_p) < 4 or len(f_p) < 4:
            diffs[b] = 0.0
            continue

        rho_m_p, _ = spearmanr(m_p[param], m_p[target])
        rho_f_p, _ = spearmanr(f_p[param], f_p[target])
        diffs[b] = rho_m_p - rho_f_p

    p = np.mean(np.abs(diffs) >= np.abs(obs_diff))
    return p


n_male = kept_filtered.loc[kept_filtered["sex"] == 1.0].shape[0]
n_female = kept_filtered.loc[kept_filtered["sex"] == 0.0].shape[0]

sex_cols = list(sex_corrs.keys())
n_sex = len(sex_cols)
lvef_cols = list(lvef_groups.keys())
n_lvef = len(lvef_cols)

row_data = []
all_abs_rhos = []

for pname, param in zip(pretty_names, param_order):
    rhos_lvef = [
        spearmanr(
            kept_filtered.loc[lvef_groups[c], param],
            kept_filtered.loc[lvef_groups[c], "ensemble"],
            nan_policy="omit",
        )[0]
        for c in lvef_cols
    ]
    rho_male = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 1.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 1.0, "ensemble"],
        nan_policy="omit",
    )[0]
    rho_female = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 0.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 0.0, "ensemble"],
        nan_policy="omit",
    )[0]

    pval = perm_test_diff_spearman_overall(kept_filtered, param, B=N_BOOTSTRAP)

    row_data.append(
        {
            "pname": pname,
            "rhos_lvef": rhos_lvef,
            "rho_male": rho_male,
            "rho_female": rho_female,
            "pval": pval,
        }
    )

    for r in rhos_lvef + [rho_male, rho_female]:
        if not np.isnan(r):
            all_abs_rhos.append(abs(r))

max_abs_rho = max(all_abs_rhos) if all_abs_rhos else 0.0


def fmt_rho_colored(rho, max_abs):
    if pd.isna(rho) or max_abs == 0:
        return ""
    abs_r = abs(rho)
    half = 0.4 * max_abs
    if abs_r <= half:
        level = 0
    elif abs_r >= max_abs:
        level = 50
    else:
        level = int(round(50 * (abs_r - half) / (max_abs - half)))
    return f"\\cellcolor{{green!{level}}} \\num{{{rho:.2f}}}"


rows = []
for rd in row_data:
    lvef_cells = [fmt_rho_colored(r, max_abs_rho) for r in rd["rhos_lvef"]]
    male_cell = fmt_rho_colored(rd["rho_male"], max_abs_rho)
    female_cell = fmt_rho_colored(rd["rho_female"], max_abs_rho)
    pval = rd["pval"]

    if pd.isna(pval):
        p_str = ""
    else:
        if pval < 0.001:
            p_str = r"<\num{0.001}"
        else:
            p_str = rf"\num{{{pval:.3f}}}"

    row = (
        "      "
        + " & ".join([rd["pname"]] + lvef_cells + [male_cell, female_cell, p_str])
        + r" \\"
    )
    rows.append(row)

col_spec_all = "l" + "r" * (n_lvef + n_sex + 1)

header_top = (
    "      & "
    + f"\\multicolumn{{{n_lvef}}}{{c}}{{LVEF subgroup}} & "
    + f"\\multicolumn{{{n_sex + 1}}}{{c}}{{Sex}} \\\\"
)

header_mid = "       & " + " & ".join(lvef_cols + ["Male", "Female", "p"]) + r" \\"

cmidrule_lvef = f"      \\cmidrule(lr){{2-{1 + n_lvef}}}"
cmidrule_sex = f"      \\cmidrule(lr){{{2 + n_lvef}-{1 + n_lvef + n_sex + 1}}}"

combined_tabular = (
    r"   {\scriptsize"
    f"\n   \\begin{{tabular*}}{{\\linewidth}}{{@{{\\extracolsep{{\\fill}}}}{col_spec_all}@{{}}}}\n"
    "\n      \\toprule\n"
    f"{header_top}\n"
    f"{cmidrule_lvef}\n"
    f"{cmidrule_sex}\n"
    f"{header_mid}\n"
    "      \\midrule\n" + "\n".join(rows) + "\n"
    "      \\bottomrule\n"
    "   \\end{tabular*}\n"
    "    }\n"
)

combined_tabular = combined_tabular.replace("_", " ")

latex_combined = (
    "\\begin{table*}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and AI–ECG HF score stratified by LVEF subgroup and sex, calculated on the Ahus cohort.}\n"
    "   \\label{tab:spearman_corr_lvef_sex}\n"
    f"{combined_tabular}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute $\\rho$ indicates stronger monotonic association. "
    "The $p$ column reports two-sided permutation test $p$-values for the difference in correlation between males and females, "
    "based on \\num{10000} random reassignments of sex labels. "
    "Values are capped at $<\\num{0.001}$.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table*}\n"
)

ARTICLE_TABLES_DIR = Path("review/article/tables")
for table_dir in (TABLES_DIR, ARTICLE_TABLES_DIR):
    table_dir.mkdir(parents=True, exist_ok=True)
    with open(table_dir / "spearman_table_lvef_sex_p.tex", "w") as f:
        f.write(latex_combined)

In [ ]:
# Ensure table output directory exists if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# -------------------------
# Helpers
# -------------------------
def compute_corr_for_df(df, param_cols, target_col="ensemble"):
    out = []
    for col in param_cols:
        if col in ("sex", "FOEDSELSNR"):
            continue
        corr, _ = spearmanr(df[col], df[target_col], nan_policy="omit")
        out.append((col, corr))
    return pd.DataFrame(out, columns=["parameter", "rho"])


def perm_test_diff_spearman_overall(df, param, target="ensemble", B=1000, seed=123):
    rng = np.random.default_rng(seed)

    sub = df[[param, target, "sex"]].dropna()

    m = sub[sub["sex"] == 1.0]
    f = sub[sub["sex"] == 0.0]

    if len(m) < 4 or len(f) < 4:
        return np.nan

    rho_m, _ = spearmanr(m[param], m[target])
    rho_f, _ = spearmanr(f[param], f[target])
    obs_diff = rho_m - rho_f

    diffs = np.empty(B)

    for b in range(B):
        permuted = sub.copy()
        permuted["sex"] = rng.permutation(permuted["sex"].values)

        m_p = permuted[permuted["sex"] == 1.0]
        f_p = permuted[permuted["sex"] == 0.0]

        if len(m_p) < 4 or len(f_p) < 4:
            diffs[b] = 0.0
            continue

        rho_m_p, _ = spearmanr(m_p[param], m_p[target])
        rho_f_p, _ = spearmanr(f_p[param], f_p[target])
        diffs[b] = rho_m_p - rho_f_p

    p = np.mean(np.abs(diffs) >= np.abs(obs_diff))
    return p


# -------------------------
# Coloring (original behavior)
# -------------------------
def fmt_rho_colored(rho, max_abs):
    if pd.isna(rho) or max_abs == 0:
        return ""
    abs_r = abs(rho)
    half = 0.4 * max_abs
    if abs_r <= half:
        level = 0
    elif abs_r >= max_abs:
        level = 50
    else:
        level = int(round(50 * (abs_r - half) / (max_abs - half)))
    return f"\\cellcolor{{green!{level}}} \\num{{{rho:.2f}}}"


# -------------------------
# Parameter ordering (overall |rho|)
# -------------------------
overall_corr_df = compute_corr_for_df(kept_filtered, plot_cols)

overall_corr_df = (
    overall_corr_df.assign(abs_corr=lambda d: d["rho"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

param_order = overall_corr_df["parameter"].tolist()
pretty_names = overall_corr_df["parameter"].tolist()

# -------------------------
# Groups
# -------------------------
lvef_groups = {
    r"$\leq$ 40\%": kept_filtered["LVEF"] <= 40,
    r"41–49\%": (kept_filtered["LVEF"] > 40) & (kept_filtered["LVEF"] < 50),
    r"$\geq$ 50\%": kept_filtered["LVEF"] >= 50,
}
lvef_cols = list(lvef_groups.keys())
n_lvef = len(lvef_cols)

# -------------------------
# Compute per-parameter stats + global max_abs for scaling
# -------------------------
row_data = []
all_abs_rhos = []

for pname, param in zip(pretty_names, param_order):
    # All
    rho_all = spearmanr(kept_filtered[param], kept_filtered["ensemble"], nan_policy="omit")[0]

    # LVEF subgroup
    rhos_lvef = [
        spearmanr(
            kept_filtered.loc[lvef_groups[c], param],
            kept_filtered.loc[lvef_groups[c], "ensemble"],
            nan_policy="omit",
        )[0]
        for c in lvef_cols
    ]

    # Sex
    rho_male = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 1.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 1.0, "ensemble"],
        nan_policy="omit",
    )[0]
    rho_female = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 0.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 0.0, "ensemble"],
        nan_policy="omit",
    )[0]

    # Permutation p-value (expects N_BOOTSTRAP defined upstream, e.g. 10000)
    pval = perm_test_diff_spearman_overall(kept_filtered, param, B=N_BOOTSTRAP)

    row_data.append(
        dict(
            pname=pname,
            rho_all=rho_all,
            rhos_lvef=rhos_lvef,
            rho_male=rho_male,
            rho_female=rho_female,
            pval=pval,
        )
    )

    for r in [rho_all] + rhos_lvef + [rho_male, rho_female]:
        if not np.isnan(r):
            all_abs_rhos.append(abs(r))

max_abs_rho = max(all_abs_rhos) if all_abs_rhos else 0.0

# -------------------------
# Build LaTeX rows
# -------------------------
rows = []
for rd in row_data:
    all_cell = fmt_rho_colored(rd["rho_all"], max_abs_rho)
    lvef_cells = [fmt_rho_colored(r, max_abs_rho) for r in rd["rhos_lvef"]]
    male_cell = fmt_rho_colored(rd["rho_male"], max_abs_rho)
    female_cell = fmt_rho_colored(rd["rho_female"], max_abs_rho)

    pval = rd["pval"]
    if pd.isna(pval):
        p_str = ""
    else:
        if pval < 0.001:
            p_str = r"<\num{0.001}"
        else:
            p_str = rf"\num{{{pval:.3f}}}"

    row = (
        "      "
        + " & ".join([rd["pname"], all_cell] + lvef_cells + [male_cell, female_cell, p_str])
        + r" \\"
    )
    rows.append(row)

# -------------------------
# LaTeX header/format
# columns: pname + All + LVEF(n_lvef) + Male + Female + p
# -------------------------
col_spec = "l" + "r" * (1 + n_lvef + 3)

header_top = (
    "      & "
    + "\\multicolumn{1}{c}{All} & "
    + f"\\multicolumn{{{n_lvef}}}{{c}}{{LVEF subgroup}} & "
    + "\\multicolumn{3}{c}{Sex} \\\\"
)

cmidrule_all = "      \\cmidrule(lr){2-2}"
cmidrule_lvef = f"      \\cmidrule(lr){{3-{2 + n_lvef}}}"
cmidrule_sex = f"      \\cmidrule(lr){{{3 + n_lvef}-{2 + n_lvef + 3}}}"

header_mid = "       & " + " & ".join(["All"] + lvef_cols + ["Male", "Female", "p"]) + r" \\"

combined_tabular = (
    r"   {\footnotesize"
    f"\n   \\begin{{tabular*}}{{\\linewidth}}{{@{{\\extracolsep{{\\fill}}}}{col_spec}@{{}}}}\n"
    "\n      \\toprule\n"
    f"{header_top}\n"
    f"{cmidrule_all}\n"
    f"{cmidrule_lvef}\n"
    f"{cmidrule_sex}\n"
    f"{header_mid}\n"
    "      \\midrule\n"
    + "\n".join(rows)
    + "\n"
    "      \\bottomrule\n"
    "   \\end{tabular*}\n"
    "    }\n"
)

combined_tabular = combined_tabular.replace("_", " ")

# -------------------------
# Full table wrapper with requested caption + bottom text
# -------------------------
latex_combined = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and\n"
    "AI–ECG HF score stratified by LVEF subgroup and sex.}\n"
    "   \\label{tab:spearman_corr_all_lvef_sex_p}\n"
    f"{combined_tabular}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute values are marked with green and indicate a stronger monotonic association. "
    "The $p$ column reports two-sided permutation test $p$-values for the difference in correlation between males and females, "
    "based on \\num{10000} random reassignments of sex labels.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

# -------------------------
# Save
# -------------------------
out_path = TABLES_DIR / "spearman_table_all_lvef_sex_p_esc.tex"
with open(out_path, "w") as f:
    f.write(latex_combined)

In [ ]:
# Ensure table output directory exists if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

param_order = overall_corr_df["parameter"].tolist()
pretty_names = overall_corr_df["parameter"].tolist()

# ---------- LVEF subgroups ----------
lvef_groups = {
    r"LVEF $\leq$ 40\%": kept_filtered["LVEF"] <= 40,
    r"LVEF 41–49\%": (kept_filtered["LVEF"] > 40) & (kept_filtered["LVEF"] < 50),
    r"LVEF $\geq$ 50\%": kept_filtered["LVEF"] >= 50,
}
lvef_cols = list(lvef_groups.keys())


def perm_test_diff_spearman(df, l_mask, param, target="ensemble", B=1000, seed=123):
    rng = np.random.default_rng(seed)

    sub = df.loc[l_mask, [param, target, "sex"]].dropna()

    m = sub[sub["sex"] == 1.0]
    f = sub[sub["sex"] == 0.0]

    if len(m) < 4 or len(f) < 4:
        return np.nan

    rho_m, _ = spearmanr(m[param], m[target])
    rho_f, _ = spearmanr(f[param], f[target])
    obs_diff = rho_m - rho_f

    diffs = np.empty(B)

    for b in range(B):
        # permute sex labels
        permuted = sub.copy()
        permuted["sex"] = rng.permutation(permuted["sex"].values)

        m_p = permuted[permuted["sex"] == 1.0]
        f_p = permuted[permuted["sex"] == 0.0]

        if len(m_p) < 4 or len(f_p) < 4:
            diffs[b] = 0.0
            continue

        rho_m_p, _ = spearmanr(m_p[param], m_p[target])
        rho_f_p, _ = spearmanr(f_p[param], f_p[target])
        diffs[b] = rho_m_p - rho_f_p

    p = np.mean(np.abs(diffs) >= np.abs(obs_diff))
    return p


sex_codes = [1.0, 0.0]
sex_labels = {1.0: "Male", 0.0: "Female"}

n_by_lvef_sex = {}
for l_label, l_mask in lvef_groups.items():
    for s in sex_codes:
        mask = l_mask & (kept_filtered["sex"] == s)
        n_by_lvef_sex[(l_label, s)] = int(mask.sum())

row_data = []
all_abs_rhos = []

for pname, param in zip(pretty_names, param_order):
    rhos = {}
    pvals = {}
    for l_label, l_mask in lvef_groups.items():
        for s in sex_codes:
            df_sub = kept_filtered.loc[l_mask & (kept_filtered["sex"] == s)]
            rho, _ = spearmanr(df_sub[param], df_sub["ensemble"], nan_policy="omit")
            rhos[(l_label, s)] = rho
            if not np.isnan(rho):
                all_abs_rhos.append(abs(rho))

        r_m = rhos[(l_label, 1.0)]
        r_f = rhos[(l_label, 0.0)]
        n_m = n_by_lvef_sex[(l_label, 1.0)]
        n_f = n_by_lvef_sex[(l_label, 0.0)]
        df_m = kept_filtered.loc[l_mask & (kept_filtered["sex"] == 1.0)]
        df_f = kept_filtered.loc[l_mask & (kept_filtered["sex"] == 0.0)]
        p = perm_test_diff_spearman(kept_filtered, l_mask, param, B=N_BOOTSTRAP)

        pvals[l_label] = p

    row_data.append(
        {
            "pname": pname,
            "rhos": rhos,
            "pvals": pvals,
        }
    )

max_abs_rho = max(all_abs_rhos) if all_abs_rhos else 0.0


def fmt_rho_colored(rho, max_abs):
    if pd.isna(rho) or max_abs == 0:
        return ""
    abs_r = abs(rho)
    half = 0.4 * max_abs
    if abs_r <= half:
        level = 0
    elif abs_r >= max_abs:
        level = 50
    else:
        level = int(round(50 * (abs_r - half) / (max_abs - half)))
    return f"\\cellcolor{{green!{level}}} \\num{{{rho:.2f}}}"


# ===== build LaTeX rows =====
rows = []
for rd in row_data:
    cells = []
    for l_label in lvef_cols:
        for s in sex_codes:
            rho = rd["rhos"][(l_label, s)]
            cells.append(fmt_rho_colored(rho, max_abs_rho))

        pval = rd["pvals"][l_label]
        if pd.isna(pval):
            p_str = ""
        else:
            if pval < 0.001:
                p_str = r"<\num{0.001}"
            else:
                p_str = rf"\num{{{pval:.3f}}}"
        cells.append(p_str)

    row = "      " + " & ".join([rd["pname"]] + cells) + r" \\"
    rows.append(row)

# For each LVEF: Male, Female, p  -> 3 columns
n_lvef = len(lvef_cols)
cols_per_lvef = 3
total_data_cols = n_lvef * cols_per_lvef

col_spec_all = "l" + "r" * total_data_cols

# top header: LVEF subgroups
header_top = (
    "      & "
    + " & ".join(
        [f"\\multicolumn{{{cols_per_lvef}}}{{c}}{{{label}}}" for label in lvef_cols]
    )
    + r" \\"
)

mid_cells = []
for _ in lvef_cols:
    mid_cells.extend(["Male", "Female", "p"])
header_mid = "      Parameter & " + " & ".join(mid_cells) + r" \\"

cmidrules = []
start = 2  # first data column
for i in range(n_lvef):
    end = start + cols_per_lvef - 1
    cmidrules.append(f"      \\cmidrule(lr){{{start}-{end}}}")
    start = end + 1
cmidrules_str = "\n".join(cmidrules)

combined_tabular = (
    f"   \\begin{{tabular}}{{{col_spec_all}}}\n"
    "      \\toprule\n"
    f"{header_top}\n"
    f"{cmidrules_str}\n"
    f"{header_mid}\n"
    "      \\midrule\n" + "\n".join(rows) + "\n"
    "      \\bottomrule\n"
    "   \\end{tabular}\n"
)

# avoid LaTeX issues with underscores in names
combined_tabular = combined_tabular.replace("_", " ")

latex_combined = (
    "\\begin{table*}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and AI–ECG HF score stratified by both LVEF subgroup and sex, calculated on the Ahus cohort.}\n"
    "   \\label{tab:spearman_corr_lvef_sex_6groups}\n"
    f"{combined_tabular}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute $\\rho$ indicates stronger monotonic association. "
    "Within each LVEF subgroup, the $p$ columns report two-sided permutation test $p$-values for the difference in correlation between males and females.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table*}\n"
)


ARTICLE_TABLES_DIR = Path("review/article/tables")
for table_dir in (TABLES_DIR, ARTICLE_TABLES_DIR):
    table_dir.mkdir(parents=True, exist_ok=True)
    with open(table_dir / "spearman_table_lvef_sex_6groups.tex", "w") as f:
        f.write(latex_combined)

In [ ]:
fig2, axs2 = plt.subplots(3, 5, figsize=(12, 7))
legend_handles, legend_labels = None, None

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols)):
    sub = kept_filtered[kept_filtered[column].notna()]
    sub = sub.copy()
    x = sub[column].values
    y = sub["ensemble"].values

    ind = np.argsort(x)
    x = x[ind]
    y = y[ind]

    lims = np.percentile(x, [2, 98])
    lims = PLOT_LIMS.get(column, lims)
    ax.set_xlim(lims)

    qrs = 0.25
    xg_q, qmean, qmedian, qlo, qhi = kernel_mean_quantile_band_rankx(
        x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
    )
    ax.plot(xg_q, qmedian, linewidth=2.0, color="C0", zorder=2, label="Median")
    ax.fill_between(
        xg_q,
        qlo,
        qhi,
        alpha=0.4,
        color="C0",
        zorder=1,
        label=f"{int(qrs*100)}–{int((1-qrs)*100)}% range",
        edgecolor="none",
    )

    ax.set_ylim(0, 1)
    unit = UNITS_MAPPING.get(column, "-")
    ax.set_title(f"{column.replace('_', ' ')}", fontsize=13)
    ax.set_xlabel(f"{unit}")
    if i in (0, 5, 10):
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

    ax2 = ax.twinx()
    sns.kdeplot(
        x,
        ax=ax2,
        color="gray",
        fill=True,
        alpha=0.3,
        edgecolor="gray",
        linewidth=0.0,
        label="Parameter density",
        zorder=0,
        bw_adjust=0.5,
    )

    ax2.set_yticks([])
    ax2.patch.set_visible(False)
    ax.set_zorder(ax2.get_zorder() + 1)  # bring model curve to front
    ax.patch.set_visible(False)

    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.spines["bottom"].set_visible(False)
    ax2.spines["left"].set_visible(False)
    ax2.set_ylabel("")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        legend_handles += handles2
        legend_labels += labels2

    # ax.set_xticks(xticks)
    # ax.set_xticklabels([f"\\num{{{tick:.0f}}}" for tick in xticks])
    # plt.tight_layout(rect=[0, 0, 1, 0.95])

if legend_handles:
    fig2.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        fontsize=12,
    )

fig2.text(
    0.0,
    0.5,
    "AI–ECG HF score (%)",
    va="center",
    rotation="vertical",
    fontsize=13,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(figure_output_path("rankx_quartile_bands.png"), dpi=300)
plt.savefig(figure_output_path("rankx_quartile_bands.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
fig2, axs2 = plt.subplots(3, 3, figsize=(9, 8))
legend_handles, legend_labels = None, None

plot_cols_esc = ['GLS',
 'MAPSE',
 'LVEF',
 'LAVi',
 'PASP',
 'LVMi',
 "E/e'",
 "e'",
 'CI',]
print

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols_esc)):
    sub = kept_filtered[kept_filtered[column].notna()]
    sub = sub.copy()
    x = sub[column].values
    y = sub["ensemble"].values

    ind = np.argsort(x)
    x = x[ind]
    y = y[ind]

    lims = np.percentile(x, [2, 98])
    lims = PLOT_LIMS.get(column, lims)
    ax.set_xlim(lims)

    qrs = 0.25
    xg_q, qmean, qmedian, qlo, qhi = kernel_mean_quantile_band_rankx(
        x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
    )
    ax.plot(xg_q, qmedian, linewidth=2.0, color="C0", zorder=2, label="Median")
    ax.fill_between(
        xg_q,
        qlo,
        qhi,
        alpha=0.4,
        color="C0",
        zorder=1,
        label=f"{int(qrs*100)}–{int((1-qrs)*100)}% range",
        edgecolor="none",
    )

    ax.set_ylim(0, 1)
    unit = UNITS_MAPPING.get(column, "-")
    ax.set_title(f"{column.replace('_', ' ')}", fontsize=13)
    ax.set_xlabel(f"{unit}")
    if i in (0, 3, 6):
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

    ax2 = ax.twinx()
    sns.kdeplot(
        x,
        ax=ax2,
        color="gray",
        fill=True,
        alpha=0.3,
        edgecolor="gray",
        linewidth=0.0,
        label="Parameter density",
        zorder=0,
        bw_adjust=0.5,
    )

    ax2.set_yticks([])
    ax2.patch.set_visible(False)
    ax.set_zorder(ax2.get_zorder() + 1)  # bring model curve to front
    ax.patch.set_visible(False)

    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.spines["bottom"].set_visible(False)
    ax2.spines["left"].set_visible(False)
    ax2.set_ylabel("")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        legend_handles += handles2
        legend_labels += labels2

    # ax.set_xticks(xticks)
    # ax.set_xticklabels([f"\\num{{{tick:.0f}}}" for tick in xticks])
    # plt.tight_layout(rect=[0, 0, 1, 0.95])

if legend_handles:
    fig2.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        fontsize=12,
    )

fig2.text(
    0.0,
    0.5,
    "AI–ECG HF score (%)",
    va="center",
    rotation="vertical",
    fontsize=13,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(figure_output_path("rankx_quartile_bands_esc.png"), dpi=300)
plt.savefig(figure_output_path("rankx_quartile_bands_esc.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
fig2, axs2 = plt.subplots(3, 5, figsize=(12, 7))

# Define how to map gender values to labels and colors
sex_info = {
    1: {"label": "Male", "color": "C0"},
    0: {"label": "Female", "color": "C1"},
}

legend_handles, legend_labels = None, None

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols)):
    sub_all = kept_filtered[kept_filtered[column].notna()].copy()

    # display limits (1st–99th pct) based on the full sample
    x_all = sub_all[column].to_numpy()
    lims = np.percentile(x_all, [1, 99])
    ax.set_xlim(lims)
    ax.set_ylim(0, 1)
    ax.set_title(column.replace("_", " "))
    ax.set_xlabel("")

    if i in (0, 5, 10):
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

    ax2 = ax.twinx()
    sns.kdeplot(
        x_all,
        ax=ax2,
        color="gray",
        fill=True,
        alpha=0.3,
        edgecolor="gray",
        linewidth=0.0,
        label="Parameter density",
        zorder=0,
        bw_adjust=0.5,
    )
    ax2.set_yticks([])
    ax2.patch.set_visible(False)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

    # remove spines of the histogram axis
    for spine in ("top", "right", "bottom", "left"):
        ax2.spines[spine].set_visible(False)
    # remove top/right spines of main axis
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # --- Sex-specific curves ---
    for sex_val, info in sex_info.items():
        sub = sub_all[sub_all["sex"] == sex_val]
        if sub.empty:
            continue

        x = sub[column].to_numpy()
        y = sub["ensemble"].to_numpy()  # AI-ECG HF score in [0,1]
        y_diag = sub[
            "Heart failure"
        ].to_numpy()  # ground-truth labels (0/1), unused here

        # sort by x
        ind = np.argsort(x)
        x = x[ind]
        y = y[ind]
        y_diag = y_diag[ind]

        # 98% central band around median
        # qrs = 0.49
        # xg_q, qlo, qhi = kernel_quantile_band_rankx(
        #     x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
        # )
        # qm = (qlo + qhi) / 2.0
        # median curve

        # 50% central band
        qrs = 0.25
        xg_q, qmean, qmedian, qlo, qhi = kernel_mean_quantile_band_rankx(
            x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
        )
        ax.plot(
            xg_q,
            qmedian,
            linewidth=2.0,
            color=info["color"],
            zorder=2,
            label=f"{info['label']} median",
        )
        ax.fill_between(
            xg_q,
            qlo,
            qhi,
            alpha=0.25,
            color=info["color"],
            zorder=1,
            edgecolor="none",
            label=f"{info['label']} {int(qrs*100)}–{int((1-qrs)*100)}% range",
        )

    # collect legend info only once
    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        legend_handles += handles2
        legend_labels += labels2

# single legend outside the grid
if legend_handles:
    fig2.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=5,
        frameon=False,
        fontsize=11,
    )

# set y label for the entire figure
fig2.text(
    0.0,
    0.5,
    "AI–ECG HF score (%)",
    va="center",
    rotation="vertical",
    fontsize=12,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.savefig(figure_output_path("rankx_quartile_bands_sex_split.png"), dpi=300)
# plt.savefig(figure_output_path("rankx_quartile_bands_sex_split.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
green_cmap = LinearSegmentedColormap.from_list(
    "green_diverging",
    ["#009900", "#FFFFFF", "#009900"],  # dark green → white → dark green
    N=256,
)

corr_spearman = pd.DataFrame(
    index=["ensemble"] + param_order,
    columns=["ensemble"] + param_order,
    data=np.nan,
)
for i, param1 in enumerate(["ensemble"] + param_order):
    for j, param2 in enumerate(["ensemble"] + param_order):
        # if j <= i:
        #     continue
        corr, _ = spearmanr(
            kept_filtered[param1], kept_filtered[param2], nan_policy="omit"
        )
        corr_spearman.at[param1, param2] = corr * 100  # in percent
        # corr_spearman.at[param2, param1] = corr * 100  # symmetric

plt.figure(figsize=(10, 10))
display_corr_spearman = corr_spearman.rename(
    index={"ensemble": "AI–ECG HF score (%)"},
    columns={"ensemble": "AI–ECG HF score (%)"},
)

sns.heatmap(
    display_corr_spearman, annot=True, fmt=".0f", cmap=green_cmap, vmin=-80, vmax=80, cbar=False
)
plt.title("Spearman Rank Correlation (\%)")
plt.xticks(rotation=35, ha="right")

plt.yticks(rotation=0)
plt.yticks(
    ticks=np.array(list(range(len(display_corr_spearman.index)))) + 0.5,
    labels=[label.replace("_", " ") for label in display_corr_spearman.index],
)
plt.xticks(
    ticks=np.array(list(range(len(display_corr_spearman.columns)))) + 0.5,
    labels=[label.replace("_", " ") for label in display_corr_spearman.columns],
)
plt.savefig(figure_output_path("spearman_corr_matrix.png"), dpi=300, bbox_inches="tight")
plt.savefig(figure_output_path("spearman_corr_matrix.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Ensure table output directory exists if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
ARTICLE_TABLES_DIR = Path("review/article/tables")
for table_dir in (TABLES_DIR, ARTICLE_TABLES_DIR):
    table_dir.mkdir(parents=True, exist_ok=True)

def wrap_num_in_num(s: str):
    """Wrap numbers in \\num{} for siunitx; strip thousands separators inside \\num."""

    def repl(m):
        digits = m.group(1).replace(",", "")
        return f"\\num{{{digits}}}"

    return re.sub(r"(?<![A-Za-z0-9])([0-9][0-9,\.]*)", repl, s)


def format_icd_label(s: str):
    """Keep ICD-code labels as compact text rather than siunitx numbers."""
    overrides = {
        "I10-I15": "I10–I15",
        "I20-25": "I20–I25",
        "I42,50,110": "I42,I50,I11.0",
        "I21,22,252": "I21,I22,I25.2",
    }
    text = str(s)
    return overrides.get(text, text.replace("-", "–"))


def fmt_n_pct(n, d):
    """Return 'n (p\\%)' string with escaped percent sign."""
    if d == 0 or pd.isna(d):
        return "0 (0\\%)"
    pct = 100.0 * n / d
    return f"{n:,} ({pct:.0f}\\%)"


def fmt_mean_sd(a):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    if a.empty:
        return ""
    return f"{a.mean():.1f} ({a.std(ddof=1):.1f})"


def fmt_median_iqr(a):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    if a.empty:
        return ""
    q1, q2, q3 = np.percentile(a, [25, 50, 75])
    return f"{q2:.0f} [{q1:.0f}, {q3:.0f}]"


def normalize_sex(x):
    if pd.isna(x):
        return "male"
    s = str(x).strip().lower()
    if s in {"f", "female", "k"}:
        return "female"
    if s in {"m", "male", "1", "1.0"}:
        return "male"
    if s in {"0", "0.0"}:
        return "female"
    return "male"


_REGEX_CACHE = {}


def any_match_regex(code_list, regex_list):
    if not isinstance(code_list, (list, tuple)):
        return False
    for pat in regex_list:
        if pat not in _REGEX_CACHE:
            _REGEX_CACHE[pat] = re.compile(pat)
        rx = _REGEX_CACHE[pat]
        for c in code_list:
            if isinstance(c, str) and rx.match(c):
                return True
    return False


def build_baseline_from_subdiag(
    subdiag: pd.DataFrame, n_total: int = None, cohort: pd.DataFrame = None
) -> pd.DataFrame:
    expected = [
        "PERSONID",
        "OMSORGSEPISODEID",
        "KODE",
        "TEKST",
        "DIAGNOSETID",
        "HOVEDDIAGNOSE",
        "FOEDSELSNR",
        "time",
        "age",
        "sex",
        "IS_BASELINE",
    ]
    missing = set(expected) - set(subdiag.columns)
    if missing:
        raise ValueError(f"subdiag missing columns: {missing}")

    d = subdiag.copy()
    d["KODE"] = d["KODE"].astype(str).str.upper().str.strip()
    d["DIAGNOSETID"] = pd.to_datetime(d["DIAGNOSETID"], errors="coerce")
    d["time"] = pd.to_datetime(d["time"], errors="coerce")
    d["IS_BASELINE"] = (
        pd.to_numeric(d["IS_BASELINE"], errors="coerce").fillna(0).astype(int)
    )
    d = d.dropna(subset=["time"])

    # earliest echo per patient -> index row (for age/sex)
    idx_rows = (
        d.sort_values(["PERSONID", "time"])
        .drop_duplicates(subset=["PERSONID"], keep="first")
        .loc[:, ["PERSONID", "time", "age", "sex"]]
        .rename(columns={"time": "index_time"})
    )
    idx_rows["age"] = pd.to_numeric(idx_rows["age"], errors="coerce")
    idx_rows["Sex_norm"] = idx_rows["sex"].apply(normalize_sex)
    if cohort is not None:
        demo_rows = (
            cohort.drop_duplicates(subset=["FOEDSELSNR"], keep="first")
            .loc[:, ["FOEDSELSNR", "age", "sex"]]
            .copy()
        )
        demo_rows["age"] = pd.to_numeric(demo_rows["age"], errors="coerce")
        demo_rows["Sex_norm"] = demo_rows["sex"].apply(normalize_sex)
    else:
        demo_rows = idx_rows

    cohort_ids = idx_rows["PERSONID"].unique()
    N = n_total or len(cohort_ids)

    # collect codes at/before index (IS_BASELINE) and all-time
    codes_idx = (
        d.loc[d["IS_BASELINE"] == 1]
        .groupby("PERSONID")["KODE"]
        .apply(list)
        .reindex(cohort_ids, fill_value=[])
    )
    codes_ever = (
        d.groupby("PERSONID")["KODE"].apply(list).reindex(cohort_ids, fill_value=[])
    )

    # assemble rows
    rows = []
    rows.append(("Patients, n", "", f"{N:,}", ""))

    if demo_rows["age"].notna().any():
        rows.append(("Age", "", "", ""))
        rows.append(("\\quad Years, mean (SD)", "", fmt_mean_sd(demo_rows["age"]), ""))
        rows.append(
            ("\\quad Years, median [IQR]", "", fmt_median_iqr(demo_rows["age"]), "")
        )

    s = demo_rows["Sex_norm"]
    rows.append(("Sex", "", "", ""))
    rows.append(("\\quad Female", "", fmt_n_pct(int((s == "female").sum()), N), ""))
    rows.append(("\\quad Male", "", fmt_n_pct(int((s == "male").sum()), N), ""))
    miss = int(s.isna().sum())
    if miss > 0:
        rows.append(("\\quad Missing", "", fmt_n_pct(miss, N), ""))

    rows.append(("Comorbidities", "", "", ""))
    comorbid_sorted = []
    for name, regex_list in COMORBIDITY_REGEX.items():
        n_idx = int(codes_idx.apply(lambda cs: any_match_regex(cs, regex_list)).sum())
        n_evr = int(codes_ever.apply(lambda cs: any_match_regex(cs, regex_list)).sum())
        icd_label = ", ".join(COMORBIDITY_MAP.get(name, []))
        comorbid_sorted.append((name, icd_label, n_idx, n_evr))
    comorbid_sorted.sort(key=lambda x: x[2], reverse=True)
    for name, icd_label, n_idx, n_evr in comorbid_sorted:
        rows.append(
            (f"\\quad {name}", icd_label, fmt_n_pct(n_idx, N), fmt_n_pct(n_evr, N))
        )

    return pd.DataFrame(
        rows, columns=["Characteristic", "ICD-10", "Before index", "All time"]
    )


# ---- build + LaTeX ----
baseline_df = build_baseline_from_subdiag(subdiag, n_total=len(kept_filtered), cohort=kept_filtered)

caption = "Baseline characteristics for the full echocardiography cohort: age, sex, and comorbidities recorded before the index date and across all available time."
label = "tab:baseline_full_echo"

latex = dedent(
    f"""
\\begin{{table}}[!htbp]
\\centering
\\begin{{threeparttable}}
\\caption{{{caption}}}
\\label{{{label}}}
{{\\scriptsize
\\begin{{tabular*}}{{\\linewidth}}{{@{{}}l@{{\\hspace{{0.4em}}}}l@{{\\extracolsep{{\\fill}}}}rr@{{}}}}
\\toprule
Characteristic & ICD-10 & Before index & All time \\\\
\\midrule
"""
).lstrip()

for ch, icd, v1, v2 in baseline_df.itertuples(index=False):
    latex += f"{ch} & {format_icd_label(icd)} & {wrap_num_in_num(v1)} & {wrap_num_in_num(v2)} \\\\\n"

latex += dedent(
    r"""
\bottomrule
\end{tabular*}
}
\begin{tablenotes}[para]
\footnotesize
Before index includes diagnostic codes registered before the index ECG. All time includes codes recorded at any time in the available EHR, including after the index.
\end{tablenotes}
\end{threeparttable}
\end{table}
"""
)

# drop empty rows from latex and indent all but first and last row
new_latex = []
for row in latex.splitlines():
    if row.strip() == "":
        continue
    if row.startswith("\\begin{table}") or row.startswith("\\end{table}"):
        new_latex.append(row)
    else:
        new_latex.append("\t" + row)
new_latex[-1] = new_latex[-1].lstrip()
latex = "\n".join(new_latex)

for table_dir in (TABLES_DIR, ARTICLE_TABLES_DIR):
    with open(table_dir / "baseline_table_full_echo.tex", "w") as f:
        f.write(latex)

In [ ]:
def _spearman_corr(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 2 or len(y) < 2:
        return np.nan
    rho, _ = spearmanr(x, y)
    return abs(rho)


def plot_rankx_ci_by_sex(
    df: pd.DataFrame,
    plot_cols: Sequence[str] = ("LVEF", "PASP", "IVS", "PWT"),
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    sexes: Sequence[str] = ("female", "male"),
    column_name_map: Optional[Dict[str, str]] = None,
    column_units_map: Optional[Dict[str, str]] = None,
    qs_inner: float = 0.975,
    qs_outer: float = 0.025,
    n_boot: int = 2000,
    random_state: Optional[int] = 0,
    min_n: int = 10,
    cache_dir: str = "./cache",
    cache_name: str = "rankx_ci_by_sex_ahus.csv",
) -> Tuple[plt.Figure, np.ndarray]:

    if column_name_map is None:
        column_name_map = {}
    if column_units_map is None:
        column_units_map = {}

    rng = np.random.default_rng(random_state)

    # NEW: container for saved statistics
    rows = []

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4), squeeze=False)
    axs = axs[0]

    x_positions = np.arange(len(sexes))
    legend_handles, legend_labels = None, None

    for ax_i, (ax, column) in enumerate(zip(axs, plot_cols)):
        meds = np.full(len(sexes), np.nan, dtype=float)
        los = np.full(len(sexes), np.nan, dtype=float)
        his = np.full(len(sexes), np.nan, dtype=float)
        ns = np.zeros(len(sexes), dtype=int)

        for s_idx, sex in enumerate(sexes):
            sub = (
                df.loc[df[sex_col] == sex, [column, prob_col]]
                .dropna()
                .reset_index(drop=True)
            )
            x = sub[column].to_numpy()
            y = sub[prob_col].to_numpy()
            n = len(x)
            ns[s_idx] = n

            if n < min_n:
                continue

            boot = np.empty(n_boot, dtype=float)
            for b in range(n_boot):
                idx = rng.integers(0, n, size=n)
                boot[b] = _spearman_corr(x[idx], y[idx])

            boot = boot[~np.isnan(boot)]
            if boot.size == 0:
                continue

            med = np.median(boot)  # change to np.mean(boot) if desired
            lo = np.quantile(boot, qs_outer)
            hi = np.quantile(boot, qs_inner)

            meds[s_idx] = med
            los[s_idx] = lo
            his[s_idx] = hi

            # NEW: store row
            sex_str = {0.0: "female", 1.0: "male"}[sex]
            rows.append(
                {
                    "variable": column,
                    "sex": sex_str,
                    "n": n,
                    "estimate": med,
                    "ci_low": lo,
                    "ci_high": hi,
                    "qs_outer": qs_outer,
                    "qs_inner": qs_inner,
                    "n_boot": n_boot,
                }
            )

        valid = ~np.isnan(meds)
        if np.any(valid):
            yerr = np.vstack([meds[valid] - los[valid], his[valid] - meds[valid]])
            ax.errorbar(
                x_positions[valid],
                meds[valid],
                yerr=yerr,
                fmt="o",
                capsize=4,
                label="Bootstrapped median ± CI",
            )

        ax.set_xticks(x_positions)
        ax.set_xticklabels(
            [f"{s}\n(n={ns[i]})" for i, s in enumerate(sexes)], fontsize=10
        )

        pretty = column_name_map.get(column, column)
        unit = column_units_map.get(column, "")
        ax.set_title(f"{pretty} [{unit}]" if unit else pretty, fontsize=11)

        if ax is axs[0]:
            ax.set_ylabel("Spearman |ρ|")
        else:
            ax.set_yticklabels([])
            ax.set_ylabel("")

        ax.set_ylim(0, 0.6)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if ax_i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            ncol=2,
            frameon=False,
            fontsize=11,
        )
    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.92])

    # NEW: save results
    if rows:
        cache_path = Path(cache_dir)
        cache_path.mkdir(parents=True, exist_ok=True)
        out_df = pd.DataFrame(rows)
        out_df.to_csv(cache_path / cache_name, index=False)

    return fig, axs


df_plot = kept_filtered.copy()
fig, axs = plot_rankx_ci_by_sex(
    df_plot,
    plot_cols=("GLS", "LVEF", "PASP", "TRv", "IVS", "PWT"),
    prob_col="ensemble",
    sex_col="sex",
    sexes=(0.0, 1.0),
    column_units_map={
        "LVEF": "%",
        "PASP": "mmHg",
        "IVS": "cm",
        "PWT": "cm",
        "TRv": "m/s",
    },
    n_boot=N_BOOTSTRAP,
    random_state=0,
    min_n=25,
)

plt.show()

In [ ]:
# Ensure table output directory exists if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Baseline characteristics by GLS availability
from scipy import stats

vars_to_test = [
    "age",
    "sex",
    "Hypertension",
    "COPD",
    "Ischaemic heart disease",
    "Atrial fibrillation",
    "Heart failure",
    "Myocardial infarction",
]

baseline_by_gls = kept_filtered.copy()
baseline_by_gls["GLS_missing_group"] = np.where(
    baseline_by_gls["GLS"].isna(), "GLS missing", "GLS present"
)


def is_discrete_var(s, max_unique=10):
    x = s.dropna()
    if x.empty:
        return False
    if (
        pd.api.types.is_bool_dtype(x)
        or pd.api.types.is_object_dtype(x)
        or pd.api.types.is_categorical_dtype(x)
    ):
        return True
    if pd.api.types.is_numeric_dtype(x):
        unique_vals = x.unique()
        integer_like = np.all(np.isclose(unique_vals, np.round(unique_vals)))
        return len(unique_vals) <= max_unique and integer_like
    return False


def fmt_mean_sd_for_gls_table(s):
    x = pd.to_numeric(s, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    return f"{x.mean():.1f} ± {x.std(ddof=1):.1f}"


def fmt_count_pct_for_gls_table(s, level=1):
    x = s.dropna()
    if len(x) == 0:
        return "NA"
    if pd.api.types.is_numeric_dtype(x):
        n = (x == level).sum()
    else:
        n = x.astype(str).str.lower().isin(["1", "true", "yes", "male", "m"]).sum()
    pct = 100 * n / len(x)
    return f"{n} ({pct:.0f}%)"


def continuous_pvalue(a, b):
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return stats.ttest_ind(a, b, equal_var=False, nan_policy="omit").pvalue


def discrete_pvalue(a, b):
    tmp = pd.DataFrame(
        {
            "group": ["GLS present"] * len(a) + ["GLS missing"] * len(b),
            "value": pd.concat([a, b], axis=0),
        }
    ).dropna()
    if tmp["group"].nunique() < 2 or tmp["value"].nunique() < 2:
        return np.nan
    tab = pd.crosstab(tmp["group"], tmp["value"])
    if tab.shape == (2, 2):
        try:
            return stats.fisher_exact(tab)[1]
        except Exception:
            return np.nan
    try:
        return stats.chi2_contingency(tab)[1]
    except Exception:
        return np.nan


def format_pvalue(p):
    if pd.isna(p):
        return "NA"
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def latex_escape_text(value):
    text = str(value).replace("_", " ").replace("%", "\\%")
    text = text.replace("±", r"$\pm$")
    if text == "<0.001":
        return r"$<0.001$"
    return text


rows = []
rows.append(
    {
        "variable": "N",
        "type": "count",
        "GLS present": f"{int(baseline_by_gls['GLS'].notna().sum())}",
        "GLS missing": f"{int(baseline_by_gls['GLS'].isna().sum())}",
        "p": "",
    }
)

for var in vars_to_test:
    if var not in baseline_by_gls.columns:
        rows.append(
            {
                "variable": var,
                "type": "missing column",
                "GLS present": "NA",
                "GLS missing": "NA",
                "p": "NA",
            }
        )
        continue

    present = baseline_by_gls.loc[baseline_by_gls["GLS"].notna(), var]
    missing = baseline_by_gls.loc[baseline_by_gls["GLS"].isna(), var]
    discrete = is_discrete_var(baseline_by_gls[var])

    if discrete:
        p = discrete_pvalue(present, missing)
        rows.append(
            {
                "variable": "Male sex" if var == "sex" else var,
                "type": "discrete",
                "GLS present": fmt_count_pct_for_gls_table(present),
                "GLS missing": fmt_count_pct_for_gls_table(missing),
                "p": format_pvalue(p),
            }
        )
    else:
        p = continuous_pvalue(present, missing)
        rows.append(
            {
                "variable": var,
                "type": "continuous",
                "GLS present": fmt_mean_sd_for_gls_table(present),
                "GLS missing": fmt_mean_sd_for_gls_table(missing),
                "p": format_pvalue(p),
            }
        )

table1 = pd.DataFrame(rows)

baseline_gls_latex_rows = []
for _, row in table1.iterrows():
    baseline_gls_latex_rows.append(
        "      "
        + " & ".join(
            [
                latex_escape_text(row["variable"]),
                latex_escape_text(row["GLS present"]),
                latex_escape_text(row["GLS missing"]),
                latex_escape_text(row["p"]),
            ]
        )
        + r" \\" 
    )

baseline_gls_latex = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Baseline characteristics stratified by GLS availability.}\n"
    "   \\label{tab:baseline_gls_availability}\n"
    "   \\begin{tabular}{lrrr}\n"
    "      \\toprule\n"
    "      Characteristic & GLS present & GLS missing & $p$ \\\\\n"
    "      \\midrule\n"
    + "\n".join(baseline_gls_latex_rows)
    + "\n      \\bottomrule\n"
    "   \\end{tabular}\n"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Continuous variables are mean $\\pm$ SD. Binary variables are shown as $n$ (\\%). The $p$ column uses Welch's $t$-test for continuous variables and Fisher's exact or chi-square tests for discrete variables.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

with open(TABLES_DIR / "baseline_table_gls_availability.tex", "w") as f:
    f.write(baseline_gls_latex)

table1


In [ ]:
# Ensure table output directory exists if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Spearman correlation table restricted to patients with available GLS
gls_present = kept_filtered.loc[kept_filtered["GLS"].notna()].copy()
gls_param_cols = [
    c
    for c in PARAMETERS
    if c not in ("sex", "age", "FOEDSELSNR") and c in gls_present.columns
]

rng_gls_spearman = np.random.default_rng(42)
gls_corr_results = []

for col in gls_param_cols:
    x = pd.to_numeric(gls_present[col], errors="coerce").to_numpy()
    y = pd.to_numeric(gls_present["ensemble"], errors="coerce").to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    n = x.size

    if n < 3:
        continue

    corr, _ = spearmanr(x, y)
    boot_corrs = []
    for _ in range(N_BOOTSTRAP):
        idx = rng_gls_spearman.integers(0, n, size=n)
        bcorr, _ = spearmanr(x[idx], y[idx])
        if np.isfinite(bcorr):
            boot_corrs.append(bcorr)

    boot_corrs = np.asarray(boot_corrs)
    ci_low = np.percentile(boot_corrs, 2.5)
    ci_high = np.percentile(boot_corrs, 97.5)
    gls_corr_results.append((col, n, corr, ci_low, ci_high))

gls_corr_df = pd.DataFrame(
    gls_corr_results,
    columns=["parameter", "n", "spearman_corr", "ci_low", "ci_high"],
)

gls_corr_df = (
    gls_corr_df.assign(abs_corr=lambda d: d["spearman_corr"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

gls_spearman_rows = []
for _, row in gls_corr_df.iterrows():
    gls_spearman_rows.append(
        "      "
        f"{row['parameter']} & "
        f"\\num{{{int(row['n'])}}} & "
        f"\\num{{{row['spearman_corr']:.2f}}} & "
        f"\\num{{{row['ci_low']:.2f}}} & "
        f"\\num{{{row['ci_high']:.2f}}} \\\\" 
    )

body = "\n".join(gls_spearman_rows) + "\n"

gls_spearman_tabular = (
    r"   {\footnotesize"
    "\n   \\begin{tabular*}{\\linewidth}{@{\\extracolsep{\\fill}}lcrrr@{}}\n"
    "      \\toprule\n"
    "      Parameter & $n$ & $Spearman \\rho$ & "
    " \\multicolumn{2}{c}{95\\% CI} \\\\\n"
    "      & & & Low & High \\\\\n"
    "      \\midrule\n"
    f"{body}"
    "      \\bottomrule\n"
    "   \\end{tabular*}\n"
    "   }\n"
)

gls_spearman_latex = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between AI–ECG HF score and echocardiographic parameters among patients with available GLS, calculated on the Ahus cohort.}\n"
    "   \\label{tab:spearman_corr_gls_present}\n"
    f"{gls_spearman_tabular.replace('_', ' ')}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item $n$ is the number of pairs used for each correlation. Higher absolute $\\rho$ indicates stronger monotonic association. Values are 95\\% bootstrap confidence intervals (\\num{10000} resamples).\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

with open(TABLES_DIR / "spearman_table_gls_present.tex", "w") as f:
    f.write(gls_spearman_latex)

gls_corr_df


In [ ]:
# Ensure table output directories exist if this cell is run independently.
from pathlib import Path

TABLES_DIR = Path("tables")
ARTICLE_TABLES_DIR = Path("review/article/tables")
for table_dir in [TABLES_DIR, ARTICLE_TABLES_DIR]:
    table_dir.mkdir(parents=True, exist_ok=True)

# Adjusted GLS association using partial Spearman rank correlation.
# This preserves the rank-correlation interpretation of the primary analysis.
from scipy import stats


def rank_percentile(s):
    return pd.Series(s).rank(method="average", pct=True).to_numpy() * 100


def residualize(y, covariates):
    x = np.column_stack([np.ones(len(covariates)), covariates.to_numpy(dtype=float)])
    beta, *_ = np.linalg.lstsq(x, y.astype(float), rcond=None)
    return y - x @ beta


def covariate_matrix(d, covariates):
    cov = pd.DataFrame(index=d.index)
    for col in covariates:
        if col == "age":
            cov["age_rank"] = rank_percentile(d[col])
        else:
            cov[col] = d[col].astype(float).to_numpy()
    return cov


def partial_spearman_value(d, covariates):
    y_rank = rank_percentile(d["ensemble"])
    gls_rank = rank_percentile(d["GLS"])
    cov = covariate_matrix(d, covariates)
    y_resid = residualize(y_rank, cov)
    gls_resid = residualize(gls_rank, cov)
    return float(stats.pearsonr(y_resid, gls_resid).statistic)


def adjusted_gls_partial_spearman(data, covariates, n_bootstrap=N_BOOTSTRAP, seed=42):
    d = data[["ensemble", "GLS", *covariates]].dropna().copy()
    rho = partial_spearman_value(d, covariates)

    rng_partial = np.random.default_rng(seed)
    boot = []
    for _ in range(n_bootstrap):
        idx = rng_partial.integers(0, len(d), size=len(d))
        boot_rho = partial_spearman_value(d.iloc[idx].reset_index(drop=True), covariates)
        if np.isfinite(boot_rho):
            boot.append(boot_rho)
    ci_low, ci_high = np.percentile(np.asarray(boot), [2.5, 97.5])

    # P value from the residual correlation; uncertainty is reported with bootstrap CI.
    y_rank = rank_percentile(d["ensemble"])
    gls_rank = rank_percentile(d["GLS"])
    cov = covariate_matrix(d, covariates)
    p_value = stats.pearsonr(residualize(y_rank, cov), residualize(gls_rank, cov)).pvalue
    return len(d), rho, float(ci_low), float(ci_high), float(p_value)


partial_spearman_specs = [
    ("Age and sex", ["age", "sex"]),
    (
        "+ Comorbidities",
        [
            "age",
            "sex",
            "Atrial fibrillation",
            "COPD",
            "Heart failure",
            "Hypertension",
            "Ischaemic heart disease",
            "Myocardial infarction",
        ],
    ),
]

adjusted_gls_rows = []
for label, covariates in partial_spearman_specs:
    n, rho, ci_low, ci_high, p_value = adjusted_gls_partial_spearman(
        kept_filtered, covariates
    )
    adjusted_gls_rows.append(
        {
            "model": label,
            "n": n,
            "partial_rho": rho,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p": p_value,
        }
    )

adjusted_gls_df = pd.DataFrame(adjusted_gls_rows)
adjusted_gls_df.to_csv(TABLES_DIR / "adjusted_gls_partial_spearman.csv", index=False)


def format_p_value(p):
    return "$<0.001$" if p < 0.001 else f"{p:.3f}"


adjusted_gls_latex_rows = []
for _, row in adjusted_gls_df.iterrows():
    adjusted_gls_latex_rows.append(
        "      "
        f"{row['model']} & "
        f"\\num{{{int(row['n'])}}} & "
        f"\\num{{{row['partial_rho']:.2f}}} & "
        f"\\num{{{row['ci_low']:.2f}}} & "
        f"\\num{{{row['ci_high']:.2f}}} & "
        f"{format_p_value(row['p'])} \\\\"
    )

adjusted_gls_latex = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Adjusted partial Spearman rank correlations between GLS and the AI–ECG HF prediction score.}\n"
    "   \\label{tab:adjusted_gls_partial_spearman}\n"
    "   {\\footnotesize\n"
    "   \\begin{tabular*}{\\linewidth}{@{\\extracolsep{\\fill}}lrrrrr@{}}\n"
    "      \\toprule\n"
    "      Adjustment set & $n$ & Partial $\\rho$ & \\multicolumn{2}{c}{95\\% CI} & $p$ \\\\\n"
    "      & & & Low & High & \\\\\n"
    "      \\midrule\n"
    + "\n".join(adjusted_gls_latex_rows)
    + "\n      \\bottomrule\n"
    "   \\end{tabular*}\n"
    "   }\n"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Partial $\\rho$ is the correlation between ranked AI–ECG and ranked GLS residuals after adjustment. ``+ Comorbidities'' additionally adjusts for baseline atrial fibrillation, COPD, heart failure, hypertension, ischaemic heart disease, and myocardial infarction.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

for table_dir in [TABLES_DIR, ARTICLE_TABLES_DIR]:
    table_dir.mkdir(parents=True, exist_ok=True)
    with open(table_dir / "adjusted_gls_partial_spearman.tex", "w") as f:
        f.write(adjusted_gls_latex)

adjusted_gls_df


In [ ]:
kept_filtered.columns

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = kept_filtered.copy()
df["af"] = df["Atrial fibrillation"]
df["mi"] = df["Myocardial infarction"]
df["ihd"] = df["Ischaemic heart disease"]
df["hf"] = df["Heart failure"]
outcome = "ensemble"   # change this to your outcome
predictors = ["LVEF", "age", "GLS", "COPD", "af", "mi", "Hypertension", "ihd", "sex", "hf", "TAPSE", "MAPSE"]

# df = df[df["mi"] == 1]
d = df[[outcome] + predictors].dropna().copy()


# Convert variables to percentiles from 0 to 100
for v in [outcome] + predictors:
    d[f"{v}_pct"] = d[v].rank(pct=True) * 100

model = smf.ols(
    f"{outcome}_pct ~ GLS_pct",
    data=d
).fit(cov_type="HC3")
print(model.summary())

model = smf.ols(
    f"{outcome}_pct ~ GLS_pct + age_pct",
    data=d
).fit(cov_type="HC3")
print(model.summary())

model = smf.ols(
    f"{outcome}_pct ~ GLS_pct + mi + af + age_pct + COPD + Hypertension + ihd + sex + hf",
    data=d
).fit(cov_type="HC3")
print(model.summary())

model = smf.ols(
    f"{outcome}_pct ~ GLS_pct + age_pct",
    data=d
).fit(cov_type="HC3")
print(model.summary())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

params = plot_cols

df = kept_filtered.copy()

# Keep only columns that exist
params = [p for p in params if p in df.columns]

# Pairwise non-missing N matrix
n_matrix = pd.DataFrame(index=params, columns=params, dtype=int)

for row_var in params:
    for col_var in params:
        n_matrix.loc[row_var, col_var] = df[[row_var, col_var]].notna().all(axis=1).sum()

# Plot
fig, ax = plt.subplots(figsize=(0.65 * len(params), 0.65 * len(params)))

im = ax.imshow(n_matrix.astype(float), aspect="auto")

ax.set_xticks(np.arange(len(params)))
ax.set_yticks(np.arange(len(params)))

ax.set_xticklabels(params, rotation=90)
ax.set_yticklabels(params)

# Write N in each cell
for i in range(len(params)):
    for j in range(len(params)):
        value = int(n_matrix.iloc[i, j])
        ax.text(
            j, i, str(value),
            ha="center", va="center",
            fontsize=7
        )

ax.set_title("Pairwise non-missing N matrix")
fig.colorbar(im, ax=ax, label="N non-missing for both variables")

plt.tight_layout()
plt.show()

# Optional: display the underlying table
n_matrix

In [ ]:
ccc = ['GLS',
 'LVEF',
 'LAVi',
 'PASP',
 'LVMi',
 "E/e'",
 "e'",
 'TAPSE',
 'LVEDVi',
 'CI',
 'A',
 'E/A',
 'Ee',
 'e'
 ]

## SL12 rhythm and baseline-HF subgroup subanalysis

These exploratory subgroup analyses use the ECG rhythm statement from the SL12 index when it can be joined to the selected ECG. The ECG feature grouping is intentionally coarse and non-exclusive: atrial fibrillation/flutter, conduction block, and left ventricular hypertrophy (LVH). ECGs without any of these machine-measured features form the normal/none class. The baseline-HF analysis uses the existing pre-ECG heart failure comorbidity flag.


In [ ]:
# Exploratory ECG-rhythm subanalysis using the SL12 index.
SL12_JOIN_VERSION = "collapse-duplicate-rhythm-keys-2026-06-29"
print(f"SL12 join code version: {SL12_JOIN_VERSION}")
# The notebook keeps local paths in paths.yaml; set sl12_index there.
# Expected SL12 columns: timestamp, pid, path, auto_text, auto_text_has_af.
SL12_INDEX_PATH = PATHS["sl12_index"]
sl12_index = pd.read_csv(SL12_INDEX_PATH)

required_sl12_cols = {"timestamp", "auto_text"}
missing_sl12_cols = required_sl12_cols - set(sl12_index.columns)
if missing_sl12_cols:
    raise ValueError(f"SL12 index is missing required columns: {sorted(missing_sl12_cols)}")

sl12_index = sl12_index.copy()
sl12_index["timestamp_dt"] = pd.to_datetime(
    sl12_index["timestamp"].astype(str), format="%Y%m%d%H%M%S", errors="coerce"
)
sl12_index["auto_text_norm"] = sl12_index["auto_text"].fillna("").astype(str).str.lower()
sl12_index["rhythm_has_flimmer_flutter"] = sl12_index["auto_text_norm"].str.contains(
    r"flimmer|flutter", regex=True, na=False
)
sl12_index["rhythm_has_blokk"] = sl12_index["auto_text_norm"].str.contains(
    r"blokk", regex=True, na=False
)
sl12_index["rhythm_has_lvh"] = (
    sl12_index["auto_text_norm"].str.contains(r"venstre", regex=True, na=False)
    & sl12_index["auto_text_norm"].str.contains(r"hypertrofi", regex=True, na=False)
)

# Recreate the nearest ECG match while preserving all available prediction-file columns.
# This allows joining to the SL12 ECG index if hf_predictions contains an ECG-level key.
ecg_cols_for_join = [c for c in df.columns if c not in echo_first.columns or c in {"FOEDSELSNR", "time"}]
df_times_full = df.dropna(subset=["FOEDSELSNR", "time"])[ecg_cols_for_join].copy()

cand_full = echo_first[["FOEDSELSNR", "SeriesDate"]].merge(
    df_times_full, on="FOEDSELSNR", how="left"
)
cand_full["abs_diff"] = (cand_full["time"] - cand_full["SeriesDate"]).abs()
idx_full = cand_full.dropna(subset=["time"]).groupby("FOEDSELSNR")["abs_diff"].idxmin()
nearest_full = cand_full.loc[idx_full].copy()

def _ecg_path_key(value):
    """Normalise ECG paths so absolute and relative SL12 paths can be joined."""
    if pd.isna(value):
        return np.nan
    parts = str(value).replace("\\", "/").split("/")
    return "/".join(parts[-2:]) if len(parts) >= 2 else parts[-1]


def _age_key(value):
    """Normalise ages such as 062Y and numeric ages to comparable integer strings."""
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d+", str(value))
    if not match:
        return np.nan
    return str(int(match.group(0)))


def _sex_key(value):
    """Normalise common sex encodings to M/F."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"m", "male", "1", "1.0"}:
        return "M"
    if text in {"f", "female", "0", "0.0"}:
        return "F"
    return text.upper()


def _has_complete_key(left: pd.DataFrame, right: pd.DataFrame, keys: Sequence[str]) -> bool:
    """Require at least one complete key on both sides before selecting a join strategy."""
    return bool(left[list(keys)].notna().all(axis=1).any() and right[list(keys)].notna().all(axis=1).any())


rhythm_join_keys = []
if "path" in nearest_full.columns and "path" in sl12_index.columns:
    nearest_full["ecg_path_key"] = nearest_full["path"].map(_ecg_path_key)
    sl12_index["ecg_path_key"] = sl12_index["path"].map(_ecg_path_key)
    if _has_complete_key(nearest_full, sl12_index, ["ecg_path_key"]):
        rhythm_join_keys = ["ecg_path_key"]

if not rhythm_join_keys and {"pid", "timestamp"}.issubset(nearest_full.columns) and {"pid", "timestamp"}.issubset(sl12_index.columns):
    nearest_full["timestamp_key"] = nearest_full["timestamp"].astype(str)
    sl12_index["timestamp_key"] = sl12_index["timestamp"].astype(str)
    if _has_complete_key(nearest_full, sl12_index, ["pid", "timestamp_key"]):
        rhythm_join_keys = ["pid", "timestamp_key"]

if not rhythm_join_keys and {"age", "sex"}.issubset(nearest_full.columns) and {"age", "sex"}.issubset(sl12_index.columns):
    # hf_predictions does not always expose an ECG path. Timestamp alone is not unique in SL12,
    # so use age and sex as additional ECG-index keys before falling back further.
    nearest_full["timestamp_dt"] = pd.to_datetime(nearest_full["time"], errors="coerce")
    nearest_full["age_key"] = nearest_full["age"].map(_age_key)
    nearest_full["sex_key"] = nearest_full["sex"].map(_sex_key)
    sl12_index["age_key"] = sl12_index["age"].map(_age_key)
    sl12_index["sex_key"] = sl12_index["sex"].map(_sex_key)
    if _has_complete_key(nearest_full, sl12_index, ["timestamp_dt", "age_key", "sex_key"]):
        rhythm_join_keys = ["timestamp_dt", "age_key", "sex_key"]

if not rhythm_join_keys and "timestamp" in nearest_full.columns and "timestamp" in sl12_index.columns:
    nearest_full["timestamp_key"] = nearest_full["timestamp"].astype(str)
    sl12_index["timestamp_key"] = sl12_index["timestamp"].astype(str)
    if _has_complete_key(nearest_full, sl12_index, ["timestamp_key"]):
        rhythm_join_keys = ["timestamp_key"]

if not rhythm_join_keys:
    # Last fallback: exact ECG time only. This is only acceptable when SL12 timestamps are unique.
    nearest_full["timestamp_dt"] = pd.to_datetime(nearest_full["time"], errors="coerce")
    rhythm_join_keys = ["timestamp_dt"]

duplicated_index_keys = int(sl12_index.duplicated(rhythm_join_keys, keep=False).sum())
print(f"SL12 rhythm join keys: {rhythm_join_keys}")
print(f"Duplicated SL12 join keys: {duplicated_index_keys}")
if duplicated_index_keys:
    print(
        "Collapsing duplicate SL12 rows within each join key by rhythm flags; "
        "use path or pid+timestamp in hf_predictions for exact ECG-level matching if available."
    )

sl12_cols_to_join = rhythm_join_keys + [
    "auto_text",
    "auto_text_has_af",
    "rhythm_has_flimmer_flutter",
    "rhythm_has_blokk",
    "rhythm_has_lvh",
]
sl12_cols_to_join = [c for c in sl12_cols_to_join if c in sl12_index.columns]

# Some SL12 timestamps are shared by multiple ECGs. When no exact ECG-level key is
# available, collapse duplicate index rows to one rhythm classification per join key
# instead of duplicating patients in the analysis merge.
def _collapse_sl12_group(group: pd.DataFrame) -> pd.Series:
    has_af_flutter = bool(group["rhythm_has_flimmer_flutter"].fillna(False).any())
    has_blokk = bool(group["rhythm_has_blokk"].fillna(False).any())
    has_lvh = bool(group["rhythm_has_lvh"].fillna(False).any())
    auto_text_values = group.get("auto_text", pd.Series(dtype=object)).dropna().astype(str).unique()
    collapsed = {key: group.iloc[0][key] for key in rhythm_join_keys}
    collapsed.update(
        {
            "auto_text": " | ".join(auto_text_values[:3]),
            "auto_text_has_af": bool(group.get("auto_text_has_af", pd.Series(False, index=group.index)).fillna(False).any()),
            "rhythm_has_flimmer_flutter": has_af_flutter,
            "rhythm_has_blokk": has_blokk,
            "rhythm_has_lvh": has_lvh,
        }
    )
    return pd.Series(collapsed)

rhythm_lookup = pd.DataFrame(
    [
        _collapse_sl12_group(group)
        for _, group in sl12_index[sl12_cols_to_join].groupby(
            rhythm_join_keys, dropna=False, sort=False
        )
    ]
)
nearest_with_rhythm = nearest_full.merge(rhythm_lookup, on=rhythm_join_keys, how="left")

# Attach rhythm status back to the existing analysis dataframe.
rhythm_cols = [
    "FOEDSELSNR",
    "time",
    "auto_text",
    "auto_text_has_af",
    "rhythm_has_flimmer_flutter",
    "rhythm_has_blokk",
    "rhythm_has_lvh",
]
rhythm_cols = [c for c in rhythm_cols if c in nearest_with_rhythm.columns]
kept_with_rhythm = kept_filtered.merge(
    nearest_with_rhythm[rhythm_cols].drop_duplicates(subset=["FOEDSELSNR", "time"]),
    on=["FOEDSELSNR", "time"],
    how="left",
)
for _flag_col in ["rhythm_has_flimmer_flutter", "rhythm_has_blokk", "rhythm_has_lvh"]:
    kept_with_rhythm[_flag_col] = kept_with_rhythm[_flag_col].fillna(False).astype(bool)

rhythm_feature_specs = [
    ("AF/flutter", "rhythm_has_flimmer_flutter"),
    ("Conduction block", "rhythm_has_blokk"),
    ("LVH", "rhythm_has_lvh"),
]
feature_frames = []
for _label, _flag_col in rhythm_feature_specs:
    _sub = kept_with_rhythm.loc[kept_with_rhythm[_flag_col]].copy()
    _sub["rhythm_group"] = _label
    feature_frames.append(_sub)

_any_ecg_feature = kept_with_rhythm[[flag for _, flag in rhythm_feature_specs]].any(axis=1)
_normal_sub = kept_with_rhythm.loc[~_any_ecg_feature].copy()
_normal_sub["rhythm_group"] = "Normal/none"
feature_frames.insert(0, _normal_sub)
kept_rhythm_long = pd.concat(feature_frames, ignore_index=True)

kept_with_rhythm["baseline_hf_group"] = np.where(
    kept_with_rhythm["Heart failure"].astype(float) == 1,
    "Baseline HF",
    "No baseline HF",
)

print("ECG feature groups; AF/flutter, conduction block, and LVH may overlap")
print(kept_rhythm_long["rhythm_group"].value_counts(dropna=False).to_string())
print("\nBaseline HF groups")
print(kept_with_rhythm["baseline_hf_group"].value_counts(dropna=False).to_string())


In [ ]:
def subgroup_spearman_table(
    data: pd.DataFrame,
    group_col: str,
    plot_cols: Sequence[str],
    target_col: str = "ensemble",
    min_n: int = 20,
) -> pd.DataFrame:
    """Compute Spearman correlations between echo metrics and AI-ECG score by subgroup."""
    rows = []
    for group, sub in data.groupby(group_col, dropna=False):
        for metric in plot_cols:
            if metric not in sub.columns:
                continue
            x = pd.to_numeric(sub[metric], errors="coerce")
            y = pd.to_numeric(sub[target_col], errors="coerce")
            mask = x.notna() & y.notna()
            n = int(mask.sum())
            rho = np.nan
            p_value = np.nan
            if n >= min_n:
                rho, p_value = spearmanr(x[mask], y[mask])
            rows.append(
                {
                    "group": group,
                    "metric": metric,
                    "n": n,
                    "rho": rho,
                    "p": p_value,
                }
            )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["metric", "group"]).reset_index(drop=True)


def plot_rankx_by_group_density_median(
    data: pd.DataFrame,
    group_col: str,
    groups: Sequence[str],
    plot_cols: Sequence[str],
    target_col: str = "ensemble",
    group_colors: Optional[Dict[str, str]] = None,
    column_name_map: Optional[Dict[str, str]] = None,
    column_units_map: Optional[Dict[str, str]] = None,
    lims: Optional[Dict[str, Sequence[float]]] = None,
    n_grid: int = 400,
    min_n: int = 20,
    title: Optional[str] = None,
    n_rows: int = 3,
    base_kernel_const: float = 1.0,
    max_kernel_const: float = 2.2,
) -> plt.Figure:
    """Overlay subgroup-specific parameter density and median AI-ECG score curves."""
    if group_colors is None:
        palette = sns.color_palette("tab10", n_colors=len(groups))
        group_colors = {group: palette[i] for i, group in enumerate(groups)}
    if column_name_map is None:
        column_name_map = {}
    if column_units_map is None:
        column_units_map = {}
    if lims is None:
        lims = {}

    plot_cols = [c for c in plot_cols if c in data.columns]
    n_metrics = len(plot_cols)
    n_rows = min(max(1, n_rows), max(1, n_metrics))
    n_cols = int(np.ceil(n_metrics / n_rows)) if n_metrics else 1
    fig, axs = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.0 * n_cols, 2.55 * n_rows),
        squeeze=False,
    )
    axs_flat = axs.ravel()

    legend_handles, legend_labels = [], []
    for i, (ax, metric) in enumerate(zip(axs_flat, plot_cols)):
        metric_data = data.loc[data[metric].notna()].copy()
        if metric_data.empty:
            ax.set_visible(False)
            continue

        x_all = pd.to_numeric(metric_data[metric], errors="coerce").dropna().to_numpy()
        if lims.get(metric) is not None:
            ax.set_xlim(lims[metric])
        else:
            ax.set_xlim(np.percentile(x_all, [1, 99]))
        ax.set_ylim(0, 1)

        ax_density = ax.twinx()
        group_xy = []
        for group in groups:
            sub = metric_data.loc[metric_data[group_col] == group].copy()
            x = pd.to_numeric(sub[metric], errors="coerce")
            y = pd.to_numeric(sub[target_col], errors="coerce")
            mask = x.notna() & y.notna()
            x = x[mask].to_numpy()
            y = y[mask].to_numpy()
            if len(x) < min_n:
                continue
            group_xy.append((group, x, y))

        max_group_n = max((len(x) for _, x, _ in group_xy), default=min_n)
        for group, x, y in group_xy:
            order = np.argsort(x)
            x = x[order]
            y = y[order]
            color = group_colors.get(group, "0.4")
            adaptive_const = min(
                max_kernel_const,
                base_kernel_const * (max_group_n / max(len(x), 1)) ** 0.25,
            )

            xg_q, qmean, qmedian, qlo, qhi = kernel_mean_quantile_band_rankx(
                x, y, qs=(0.25, 0.75), n_grid=n_grid, const=adaptive_const
            )
            line = ax.plot(
                xg_q,
                qmedian,
                linewidth=2.0,
                color=color,
                zorder=3,
                label=f"{group} median",
            )[0]

            # Density is shown on a hidden secondary axis so each subgroup's x-distribution is visible.
            if len(np.unique(x)) > 1:
                sns.kdeplot(
                    x=x,
                    ax=ax_density,
                    color=color,
                    fill=False,
                    alpha=0.28,
                    linewidth=0.9,
                    zorder=1,
                    bw_adjust=0.65,
                )

            if i == 0:
                legend_handles.append(line)
                legend_labels.append(f"{group} median")

        ax_density.set_yticks([])
        ax_density.patch.set_visible(False)
        ax.set_zorder(ax_density.get_zorder() + 1)
        ax.patch.set_visible(False)
        for spine in ax_density.spines.values():
            spine.set_visible(False)

        ax.set_title(column_name_map.get(metric, metric.replace("_", " ")))
        ax.set_xlabel(column_units_map.get(metric, UNITS_MAPPING.get(metric, "")))
        if i % n_cols == 0:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
        else:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for ax in axs_flat[n_metrics:]:
        ax.set_visible(False)

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.965),
            ncol=min(len(legend_handles), 4),
            frameon=False,
            fontsize=12,
        )
    fig.text(0.0, 0.5, "AI–ECG HF score (%)", va="center", rotation="vertical", fontsize=13)
    plt.tight_layout(rect=[0.00, 0.02, 1, 0.91])
    return fig


In [ ]:
# ECG-feature-stratified exploratory plots and correlations.
# Reuse the main-analysis ordering and show up to 15 echo markers in three rows.
subanalysis_plot_cols = [
    c for c in plot_cols
    if c in kept_rhythm_long.columns and c not in {"FOEDSELSNR", "sex", "age"}
][:15]

rhythm_groups = ["Normal/none", "Conduction block", "LVH", "AF/flutter"]
# Okabe-Ito palette: blue for normal/none, green for conduction block, orange for LVH, vermillion for AF/flutter.
rhythm_colors = {
    "Normal/none": "#0072B2",
    "Conduction block": "#009E73",
    "LVH": "#E69F00",
    "AF/flutter": "#D55E00",
}

fig = plot_rankx_by_group_density_median(
    kept_rhythm_long,
    group_col="rhythm_group",
    groups=rhythm_groups,
    plot_cols=subanalysis_plot_cols,
    group_colors=rhythm_colors,
    lims=PLOT_LIMS,
    n_rows=3,
)
for _fmt in ("png", "pdf"):
    _out = figure_output_path(f"rankx_by_sl12_rhythm_group.{_fmt}")
    plt.savefig(_out, dpi=300, bbox_inches="tight")
    _article_out = Path("review/article/figures/results") / _out.name
    _article_out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(_article_out, dpi=300, bbox_inches="tight")
plt.show()

rhythm_corr_table = subgroup_spearman_table(
    kept_rhythm_long,
    group_col="rhythm_group",
    plot_cols=subanalysis_plot_cols,
)
print("Spearman correlations by selected-ECG feature group")
display(rhythm_corr_table)


In [ ]:
# Baseline-HF-stratified exploratory plots and correlations.
baseline_hf_groups = ["No baseline HF", "Baseline HF"]
# Okabe-Ito palette: blue for no baseline HF, vermillion for baseline HF.
baseline_hf_colors = {
    "No baseline HF": "#0072B2",
    "Baseline HF": "#D55E00",
}

fig = plot_rankx_by_group_density_median(
    kept_with_rhythm,
    group_col="baseline_hf_group",
    groups=baseline_hf_groups,
    plot_cols=subanalysis_plot_cols,
    group_colors=baseline_hf_colors,
    lims=PLOT_LIMS,
    n_rows=3,
)
for _fmt in ("png", "pdf"):
    _out = figure_output_path(f"rankx_by_baseline_hf_group.{_fmt}")
    plt.savefig(_out, dpi=300, bbox_inches="tight")
    _article_out = Path("review/article/figures/results") / _out.name
    _article_out.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(_article_out, dpi=300, bbox_inches="tight")
plt.show()

baseline_hf_corr_table = subgroup_spearman_table(
    kept_with_rhythm,
    group_col="baseline_hf_group",
    plot_cols=subanalysis_plot_cols,
)
print("Spearman correlations by baseline heart failure group")
display(baseline_hf_corr_table)
